# Imports

In [1750]:
from math import log as log
from math import inf as inf
from math import floor as floor
import numpy as np

# Constantes Globais

In [1751]:
p = 0.5
N = 500 #definido arbitrariamente
MAXLEVEL = int(log(N, 1/p))

# TASKs

In [1752]:
class Task:
    def __init__(self, atributos: dict):
        #att = ["task_id", "start_time", "end_times"]
        self.atributos = atributos

    def __getitem__(self, chave):
        return self.atributos[chave]

    def __setitem__(self, chave, valor):
        self.atributos[chave] = valor

# Skip List - ATIVIDADE B)

In [1753]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [1754]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [1755]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca
    y = self.cabeca
    output = [None]*(self.tam+1)
    nvly = self.nivel_atual

    for i in range(self.nivel_atual, -1, -1):
      while y.frente[i] != self.NULL and y.frente[i].conteudo['task_id'] <= chaveBuscaB:
        y = y.frente[i]
      nvly = i

    for j in range(nvly,-1,-1):
      while x.frente[j] != self.NULL and x.frente[j].conteudo['task_id'] < chaveBuscaA:
        x = x.frente[j]

    x = x.frente[0]
    c = 0

    if x.conteudo['task_id'] >= chaveBuscaA and y.conteudo['task_id'] <= chaveBuscaB:
      while x.conteudo['task_id'] <= y.conteudo['task_id']:
        output[c] = x.conteudo
        x = x.frente[0]
        c+=1

    return output

## EXEMPLO DE USO - SKIP LIST

In [1756]:
skip = Skip_list()

### TESTE INSERT

In [1757]:
#c = ["task_id", "start_time", "end_times"]
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(0,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  skip.insert(x,y,z)


### VISUALIZAÇÃO SKIPLIST

In [1758]:
def imprime_linear(skip:Skip_list):
  x = skip.cabeca.frente[0]
  while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

In [1759]:
def imprime_estrutura(skip:Skip_list):
  x = skip.cabeca.frente[0]
  for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

In [1760]:
imprime_linear(skip)

{'task_id': 13, 'start_time': 170, 'end_time': 1712}
{'task_id': 14, 'start_time': 135, 'end_time': 1163}
{'task_id': 36, 'start_time': 827, 'end_time': 644}
{'task_id': 38, 'start_time': 91, 'end_time': 1016}
{'task_id': 61, 'start_time': 54, 'end_time': 1106}
{'task_id': 65, 'start_time': 174, 'end_time': 772}
{'task_id': 83, 'start_time': 96, 'end_time': 979}
{'task_id': 94, 'start_time': 739, 'end_time': 183}


In [1761]:
imprime_estrutura(skip)

--- Nível 4 ---
65 -> NULL
--- Nível 3 ---
14 -> 65 -> 83 -> NULL
--- Nível 2 ---
14 -> 65 -> 83 -> NULL
--- Nível 1 ---
13 -> 14 -> 65 -> 83 -> 94 -> NULL
--- Nível 0 ---
13 -> 14 -> 36 -> 38 -> 61 -> 65 -> 83 -> 94 -> NULL


## TESTE DELETE

In [1762]:
skip.delete(5)

### VISUALIZAÇÃO SKIPLIST

In [1763]:
imprime_linear(skip)

{'task_id': 13, 'start_time': 170, 'end_time': 1712}
{'task_id': 14, 'start_time': 135, 'end_time': 1163}
{'task_id': 36, 'start_time': 827, 'end_time': 644}
{'task_id': 38, 'start_time': 91, 'end_time': 1016}
{'task_id': 61, 'start_time': 54, 'end_time': 1106}
{'task_id': 65, 'start_time': 174, 'end_time': 772}
{'task_id': 83, 'start_time': 96, 'end_time': 979}
{'task_id': 94, 'start_time': 739, 'end_time': 183}


In [1764]:
imprime_estrutura(skip)

--- Nível 4 ---
65 -> NULL
--- Nível 3 ---
14 -> 65 -> 83 -> NULL
--- Nível 2 ---
14 -> 65 -> 83 -> NULL
--- Nível 1 ---
13 -> 14 -> 65 -> 83 -> 94 -> NULL
--- Nível 0 ---
13 -> 14 -> 36 -> 38 -> 61 -> 65 -> 83 -> 94 -> NULL


## Teste SEARCH

In [1765]:
skip.search(26)

False

### VISUALIZAÇÃO SKIPLIST

In [1766]:
imprime_linear(skip)

{'task_id': 13, 'start_time': 170, 'end_time': 1712}
{'task_id': 14, 'start_time': 135, 'end_time': 1163}
{'task_id': 36, 'start_time': 827, 'end_time': 644}
{'task_id': 38, 'start_time': 91, 'end_time': 1016}
{'task_id': 61, 'start_time': 54, 'end_time': 1106}
{'task_id': 65, 'start_time': 174, 'end_time': 772}
{'task_id': 83, 'start_time': 96, 'end_time': 979}
{'task_id': 94, 'start_time': 739, 'end_time': 183}


In [1767]:
imprime_estrutura(skip)

--- Nível 4 ---
65 -> NULL
--- Nível 3 ---
14 -> 65 -> 83 -> NULL
--- Nível 2 ---
14 -> 65 -> 83 -> NULL
--- Nível 1 ---
13 -> 14 -> 65 -> 83 -> 94 -> NULL
--- Nível 0 ---
13 -> 14 -> 36 -> 38 -> 61 -> 65 -> 83 -> 94 -> NULL


## Teste SEARCHINTERVAL

In [1768]:
r = skip.searchinterval(0,0)

In [1769]:
for i in range(len(r)):
  if r[i] != None:
    print(r[i].atributos)

### VISUALIZAÇÃO SKIPLIST

In [1770]:
imprime_linear(skip)

{'task_id': 13, 'start_time': 170, 'end_time': 1712}
{'task_id': 14, 'start_time': 135, 'end_time': 1163}
{'task_id': 36, 'start_time': 827, 'end_time': 644}
{'task_id': 38, 'start_time': 91, 'end_time': 1016}
{'task_id': 61, 'start_time': 54, 'end_time': 1106}
{'task_id': 65, 'start_time': 174, 'end_time': 772}
{'task_id': 83, 'start_time': 96, 'end_time': 979}
{'task_id': 94, 'start_time': 739, 'end_time': 183}


In [1771]:
imprime_estrutura(skip)

--- Nível 4 ---
65 -> NULL
--- Nível 3 ---
14 -> 65 -> 83 -> NULL
--- Nível 2 ---
14 -> 65 -> 83 -> NULL
--- Nível 1 ---
13 -> 14 -> 65 -> 83 -> 94 -> NULL
--- Nível 0 ---
13 -> 14 -> 36 -> 38 -> 61 -> 65 -> 83 -> 94 -> NULL


# AVL - ATIVIDADE C)

In [1772]:
class No_AVL:
  def __init__(self, esq=None, dir=None, task:Task=None):
    self.esq = esq
    self.dir = dir
    self.fator_balanceamento = 0
    self.altura = 0
    self.conteudo = task

In [1773]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['task_id']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['task_id'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['task_id']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    return output



## EXEMPLO DE USO - AVL

In [1774]:
AVL = AVL()

In [1775]:
raiz = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  raiz=AVL.insert(raiz,t)

### Teste INSERT

In [1776]:
def percursoIN_ORDER(raiz):
  if raiz == None:
    return
  percursoIN_ORDER(raiz.esq)
  print(raiz.conteudo.atributos)
  percursoIN_ORDER(raiz.dir)

### Visualização

In [1777]:
percursoIN_ORDER(raiz)

{'task_id': 10, 'start_time': 256, 'end_time': 1217}
{'task_id': 14, 'start_time': 695, 'end_time': 1573}
{'task_id': 51, 'start_time': 728, 'end_time': 1338}
{'task_id': 58, 'start_time': 26, 'end_time': 1080}
{'task_id': 59, 'start_time': 84, 'end_time': 1892}
{'task_id': 60, 'start_time': 20, 'end_time': 1919}
{'task_id': 64, 'start_time': 307, 'end_time': 1428}
{'task_id': 84, 'start_time': 526, 'end_time': 1251}


### Teste DELETE

In [1778]:
raiz=AVL.delete(raiz, 79)

### Visualização

In [1779]:
percursoIN_ORDER(raiz)

{'task_id': 10, 'start_time': 256, 'end_time': 1217}
{'task_id': 14, 'start_time': 695, 'end_time': 1573}
{'task_id': 51, 'start_time': 728, 'end_time': 1338}
{'task_id': 58, 'start_time': 26, 'end_time': 1080}
{'task_id': 59, 'start_time': 84, 'end_time': 1892}
{'task_id': 60, 'start_time': 20, 'end_time': 1919}
{'task_id': 64, 'start_time': 307, 'end_time': 1428}
{'task_id': 84, 'start_time': 526, 'end_time': 1251}


### Teste SEARCH

In [1780]:
busca=AVL.search(raiz,67)

In [1781]:
if busca:
  print(busca.conteudo.atributos)
else:
  print("FALSO")

FALSO


## Teste SEARCHINTERVAL

In [1782]:
output = []

In [1783]:
buscaIntervalo = AVL.searchinterval(raiz,0,60, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

{'task_id': 10, 'start_time': 256, 'end_time': 1217}
{'task_id': 14, 'start_time': 695, 'end_time': 1573}
{'task_id': 51, 'start_time': 728, 'end_time': 1338}
{'task_id': 58, 'start_time': 26, 'end_time': 1080}
{'task_id': 59, 'start_time': 84, 'end_time': 1892}
{'task_id': 60, 'start_time': 20, 'end_time': 1919}


# ATIVIDADE - D)
Adapted from: https://github.com/amitjoshi2724/gpi-job-scheduling
## corpus sintético

In [1784]:
import random

## Geração de Tasks - Especificação da Atividade

In [1785]:
def generate_uniform_tasks(n):

    MAX_VAL = 10**6
    tasks = [(random.randint(0, MAX_VAL), random.randint(0, MAX_VAL), id) for id in range(n)]
    tasks = [{'task_id': id,
              'start_time': min(s, e),
              'end_time': max(s, e)} for s, e, id in tasks]

    return tasks

t_uniform = generate_uniform_tasks(100)

In [1786]:
def generate_normal_tasks(n):

    K = 10**9
    sigma = K / 10

    # Truncated normal start times in [0, K]
    start_times = np.clip(np.random.normal(loc=K/2, scale=sigma, size=n), 0, K)
    durations = np.random.uniform(1.0, 10**6, size=n)
    end_times = start_times + durations
    tasks = [{'task_id': id,
              'start_time': s,
              'end_time': e} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

t_normal = generate_normal_tasks(100)

In [1787]:
def generate_zipf_tasks(n):

    K = 10**9  # Upper bound of time domain

    # Clustered start times near 0 using truncated exponential
    scale = K / 10  # Controls clustering tightness; lower = tighter near 0
    start_times = np.random.exponential(scale=scale, size=n)
    start_times = np.clip(start_times, 0, K)  # Ensure within [0, K]

    # Zipf durations, scaled and capped
    raw_zipf = np.random.zipf(a=2.0, size=n)
    durations = np.minimum(100 * raw_zipf, 10**6)
    end_times = start_times + durations

    tasks = [{'task_id': id,
              'start_time': float(s),
              'end_time': float(e)} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

t_zipf = generate_zipf_tasks(100)

## Adaptação SearchInterval SKIP LIST
A implementação anterior era mais genérica se baseando apenas no valor da chave ($task_{id}$)

Nesta implementação, adaptei para atender ao *timestamp* ($start\_time \geq chaveBuscaA$ e $chaveBuscaB \geq end\_time$)

Contudo, precisei fazer algumas correções por que a forma anterior funcionava para quando queríamos comparar $chaveBuscaA$ ou $chaveBuscaB$ apenas com $task\_id$

Nesta nova versão, descarto tudo o que usava $y$ (apontador superior para definir fim do intervalo) e uso apenas $x$ para percorrer a estrutura uma única vez e mais outra vez para gerar a saída

In [1788]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [1789]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [1790]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      print(x.conteudo.atributos)
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca.frente[0]
    output = [None]*(self.tam+1)

    c = 0
    while x != self.NULL:
      if chaveBuscaA <= x.conteudo['start_time'] and x.conteudo['end_time'] <= chaveBuscaB:
        output[c] = x.conteudo
        c+=1
      x = x.frente[0]

    return output

### Teste Novo SearchInterval

In [1791]:
s = Skip_list()

In [1792]:
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  s.insert(x,y,z)

In [1793]:
v = s.searchinterval(50,1700)

In [1794]:
for i in range(len(v)):
  if v[i] != None:
    print(v[i].atributos)

{'task_id': 3, 'start_time': 223, 'end_time': 1364}
{'task_id': 8, 'start_time': 685, 'end_time': 1283}
{'task_id': 62, 'start_time': 352, 'end_time': 1165}
{'task_id': 72, 'start_time': 430, 'end_time': 1604}
{'task_id': 75, 'start_time': 357, 'end_time': 1125}
{'task_id': 83, 'start_time': 176, 'end_time': 1483}
{'task_id': 85, 'start_time': 287, 'end_time': 1017}


#### VISUALIZAÇÃO SKIPLIST

In [1795]:
imprime_linear(s)

{'task_id': 3, 'start_time': 223, 'end_time': 1364}
{'task_id': 8, 'start_time': 685, 'end_time': 1283}
{'task_id': 62, 'start_time': 352, 'end_time': 1165}
{'task_id': 72, 'start_time': 430, 'end_time': 1604}
{'task_id': 75, 'start_time': 357, 'end_time': 1125}
{'task_id': 83, 'start_time': 176, 'end_time': 1483}
{'task_id': 85, 'start_time': 287, 'end_time': 1017}
{'task_id': 98, 'start_time': 898, 'end_time': 1774}


In [1796]:
imprime_estrutura(s)

--- Nível 3 ---
62 -> NULL
--- Nível 2 ---
62 -> 75 -> 98 -> NULL
--- Nível 1 ---
8 -> 62 -> 72 -> 75 -> 83 -> 98 -> NULL
--- Nível 0 ---
3 -> 8 -> 62 -> 72 -> 75 -> 83 -> 85 -> 98 -> NULL


## Adaptações AVL - SearchInterval
A adaptação, assim como na *skip list*, foi o uso de $start\_time$ e $end\_time$ ao invés de $task\_id$

De resto a estrutura do algoritmo se manteve a mesma, apenas mudanças nas comparações do *if*

In [1797]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['start_time']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['start_time'] and raiz.conteudo['end_time'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['end_time']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    return output

### Teste Novo SearchInterval

In [1798]:
avl = AVL()

In [1799]:
root = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  root=avl.insert(root,t)

In [1800]:
output = []
buscaIntervalo = avl.searchinterval(raiz,600,1800, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

### Visualização

In [1801]:
percursoIN_ORDER(raiz)

{'task_id': 10, 'start_time': 256, 'end_time': 1217}
{'task_id': 14, 'start_time': 695, 'end_time': 1573}
{'task_id': 51, 'start_time': 728, 'end_time': 1338}
{'task_id': 58, 'start_time': 26, 'end_time': 1080}
{'task_id': 59, 'start_time': 84, 'end_time': 1892}
{'task_id': 60, 'start_time': 20, 'end_time': 1919}
{'task_id': 64, 'start_time': 307, 'end_time': 1428}
{'task_id': 84, 'start_time': 526, 'end_time': 1251}


## Constantes

In [1802]:
w = np.random.random()
while w < 0.5:
  w = np.random.random()
p = w
N = np.random.randint(1,2**12)
MAXLEVEL = int(log(N, 1/p))

In [1803]:
p

0.9194625985346907

In [1804]:
N

913

In [1805]:
MAXLEVEL

81

# Instanciação

In [1806]:
skipList = Skip_list()
arvoreAVL = AVL()

## Distribuição Uniforme - Skip List

### INSERT

In [1807]:
t = generate_uniform_tasks(N)
for k in t:
  skipList.insert(k['task_id'], k['start_time'], k['end_time'])

In [1808]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 910389, 'end_time': 949811}
{'task_id': 1, 'start_time': 291005, 'end_time': 767835}
{'task_id': 2, 'start_time': 308217, 'end_time': 758762}
{'task_id': 3, 'start_time': 304150, 'end_time': 880374}
{'task_id': 4, 'start_time': 810717, 'end_time': 906731}
{'task_id': 5, 'start_time': 178958, 'end_time': 209272}
{'task_id': 6, 'start_time': 94496, 'end_time': 780066}
{'task_id': 7, 'start_time': 127899, 'end_time': 341636}
{'task_id': 8, 'start_time': 8490, 'end_time': 484556}
{'task_id': 9, 'start_time': 700837, 'end_time': 961397}
{'task_id': 10, 'start_time': 186631, 'end_time': 377211}
{'task_id': 11, 'start_time': 804123, 'end_time': 811711}
{'task_id': 12, 'start_time': 367556, 'end_time': 836349}
{'task_id': 13, 'start_time': 675802, 'end_time': 691791}
{'task_id': 14, 'start_time': 488007, 'end_time': 555041}
{'task_id': 15, 'start_time': 276679, 'end_time': 729414}
{'task_id': 16, 'start_time': 332885, 'end_time': 728520}
{'task_id': 17, 'start_time

In [1809]:
imprime_estrutura(skipList)

--- Nível 80 ---
548 -> NULL
--- Nível 79 ---
548 -> NULL
--- Nível 78 ---
548 -> NULL
--- Nível 77 ---
548 -> NULL
--- Nível 76 ---
548 -> NULL
--- Nível 75 ---
548 -> NULL
--- Nível 74 ---
548 -> NULL
--- Nível 73 ---
548 -> NULL
--- Nível 72 ---
548 -> NULL
--- Nível 71 ---
548 -> NULL
--- Nível 70 ---
548 -> NULL
--- Nível 69 ---
548 -> NULL
--- Nível 68 ---
548 -> NULL
--- Nível 67 ---
548 -> NULL
--- Nível 66 ---
548 -> NULL
--- Nível 65 ---
548 -> NULL
--- Nível 64 ---
548 -> 648 -> NULL
--- Nível 63 ---
548 -> 648 -> NULL
--- Nível 62 ---
548 -> 648 -> NULL
--- Nível 61 ---
548 -> 648 -> NULL
--- Nível 60 ---
548 -> 648 -> NULL
--- Nível 59 ---
548 -> 648 -> NULL
--- Nível 58 ---
437 -> 548 -> 648 -> NULL
--- Nível 57 ---
437 -> 548 -> 648 -> NULL
--- Nível 56 ---
437 -> 548 -> 648 -> 850 -> NULL
--- Nível 55 ---
437 -> 548 -> 648 -> 850 -> NULL
--- Nível 54 ---
437 -> 548 -> 648 -> 850 -> NULL
--- Nível 53 ---
437 -> 548 -> 636 -> 648 -> 850 -> NULL
--- Nível 52 ---
437 -> 548

### DELETE

In [1810]:
skipList.tam

913

In [1811]:
skipList.nivel_atual

80

In [1812]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, skipList.tam)
  skipList.delete(z)

In [1813]:
skipList.tam

537

In [1814]:
skipList.nivel_atual

64

In [1815]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 910389, 'end_time': 949811}
{'task_id': 2, 'start_time': 308217, 'end_time': 758762}
{'task_id': 3, 'start_time': 304150, 'end_time': 880374}
{'task_id': 4, 'start_time': 810717, 'end_time': 906731}
{'task_id': 5, 'start_time': 178958, 'end_time': 209272}
{'task_id': 6, 'start_time': 94496, 'end_time': 780066}
{'task_id': 7, 'start_time': 127899, 'end_time': 341636}
{'task_id': 8, 'start_time': 8490, 'end_time': 484556}
{'task_id': 10, 'start_time': 186631, 'end_time': 377211}
{'task_id': 11, 'start_time': 804123, 'end_time': 811711}
{'task_id': 12, 'start_time': 367556, 'end_time': 836349}
{'task_id': 13, 'start_time': 675802, 'end_time': 691791}
{'task_id': 14, 'start_time': 488007, 'end_time': 555041}
{'task_id': 15, 'start_time': 276679, 'end_time': 729414}
{'task_id': 16, 'start_time': 332885, 'end_time': 728520}
{'task_id': 17, 'start_time': 621185, 'end_time': 983795}
{'task_id': 19, 'start_time': 539070, 'end_time': 819082}
{'task_id': 20, 'start_ti

In [1816]:
imprime_estrutura(skipList)

--- Nível 64 ---
648 -> NULL
--- Nível 63 ---
648 -> NULL
--- Nível 62 ---
648 -> NULL
--- Nível 61 ---
648 -> NULL
--- Nível 60 ---
648 -> NULL
--- Nível 59 ---
648 -> NULL
--- Nível 58 ---
648 -> NULL
--- Nível 57 ---
648 -> NULL
--- Nível 56 ---
648 -> 850 -> NULL
--- Nível 55 ---
648 -> 850 -> NULL
--- Nível 54 ---
648 -> 850 -> NULL
--- Nível 53 ---
648 -> 850 -> NULL
--- Nível 52 ---
648 -> 850 -> NULL
--- Nível 51 ---
648 -> 850 -> NULL
--- Nível 50 ---
648 -> 850 -> NULL
--- Nível 49 ---
648 -> 850 -> NULL
--- Nível 48 ---
648 -> 850 -> NULL
--- Nível 47 ---
648 -> 850 -> NULL
--- Nível 46 ---
446 -> 648 -> 850 -> NULL
--- Nível 45 ---
346 -> 446 -> 648 -> 850 -> NULL
--- Nível 44 ---
346 -> 446 -> 648 -> 807 -> 850 -> NULL
--- Nível 43 ---
346 -> 446 -> 648 -> 683 -> 807 -> 850 -> NULL
--- Nível 42 ---
346 -> 446 -> 648 -> 683 -> 807 -> 850 -> NULL
--- Nível 41 ---
346 -> 446 -> 620 -> 648 -> 683 -> 807 -> 850 -> NULL
--- Nível 40 ---
346 -> 446 -> 620 -> 648 -> 683 -> 807 -> 

### Search

In [1817]:
for i in range(skipList.tam):
  z = np.random.randint(0, skipList.tam)
  if not skipList.search(z):
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {skipList.search(z)}\n=============================================================')

Chave:375; Resultado da Busca: False
{'task_id': 194, 'start_time': 194257, 'end_time': 317867}
{'task_id': 249, 'start_time': 591743, 'end_time': 766879}
{'task_id': 202, 'start_time': 595320, 'end_time': 722406}
{'task_id': 139, 'start_time': 4250, 'end_time': 290302}
Chave:296; Resultado da Busca: False
{'task_id': 262, 'start_time': 2580, 'end_time': 896463}
{'task_id': 521, 'start_time': 602877, 'end_time': 963448}
Chave:116; Resultado da Busca: False
Chave:235; Resultado da Busca: False
Chave:213; Resultado da Busca: False
Chave:447; Resultado da Busca: False
Chave:127; Resultado da Busca: False
Chave:334; Resultado da Busca: False
Chave:283; Resultado da Busca: False
{'task_id': 142, 'start_time': 137640, 'end_time': 292588}
Chave:184; Resultado da Busca: False
Chave:348; Resultado da Busca: False
{'task_id': 279, 'start_time': 221161, 'end_time': 888852}
{'task_id': 499, 'start_time': 36195, 'end_time': 514799}
{'task_id': 206, 'start_time': 193950, 'end_time': 955856}
Chave:49

### SEARCH INTERVAL

In [1818]:
for i in range(skipList.tam):
  a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
  b = np.random.randint(0, N*np.random.randint(1,N))
  while a > b:
      a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
      b = np.random.randint(0, N*np.random.randint(1,N))
  r = (skipList.searchinterval(a,b))
  print(f'=============================\nA={a}; B={b}')
  for k in r:
    if k is not None:
      print(k.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
{'task_id': 698, 'start_time': 46935, 'end_time': 254666}
{'task_id': 716, 'start_time': 28669, 'end_time': 261451}
{'task_id': 720, 'start_time': 168965, 'end_time': 256216}
{'task_id': 745, 'start_time': 8648, 'end_time': 49845}
{'task_id': 749, 'start_time': 119216, 'end_time': 206449}
{'task_id': 786, 'start_time': 205534, 'end_time': 272019}
{'task_id': 807, 'start_time': 109913, 'end_time': 259874}
{'task_id': 820, 'start_time': 206858, 'end_time': 255872}
{'task_id': 856, 'start_time': 17801, 'end_time': 245688}
{'task_id': 860, 'start_time': 115447, 'end_time': 173448}
{'task_id': 877, 'start_time': 27414, 'end_time': 239624}
{'task_id': 883, 'start_time': 94639, 'end_time': 137137}
{'task_id': 906, 'start_time': 93959, 'end_time': 290439}
A=51348; B=289077
{'task_id': 5, 'start_time': 178958, 'end_time': 209272}
{'task_id': 149, 'start_time': 54736, 'end_time': 224747}
{'task_id': 152, 'start_time': 144619, 'end_time':

## Distribuição Normal - Skip List

## Distribuição ZIPF - Skip List

## Distribuição Uniforme - AVL

## Distribuição Normal - AVL

## Distribuição ZIPF - AVL